In [1]:
import pandas as pd
from math import cos, radians, sqrt
from sklearn.neighbors import BallTree
import geopandas as gpd
from shapely.geometry import Polygon, MultiPolygon, Point
import numpy as np
import seaborn as sns
import seaborn.objects as so
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import plotly.express as px
import warnings
from sklearn.cluster import KMeans
from sklearn.preprocessing import MinMaxScaler
from yellowbrick.cluster import KElbowVisualizer
from scipy.stats import skew
from sklearn.preprocessing import StandardScaler
from statsmodels.stats.proportion import proportions_ztest
from scipy.stats import binomtest
from tqdm import tqdm
from scipy.stats import chi2_contingency
from scipy.stats import norm
import statsmodels.formula.api as smf
from scipy.stats import pearsonr

In [2]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 50)
pd.set_option('display.float_format', '{:.3f}'.format)
warnings.filterwarnings('ignore')

In [3]:
raw = "C:\\Users\\Taavi\\Desktop\\BPhil\\Raw data\\"
clean = "C:\\Users\\Taavi\\Desktop\\BPhil\\Clean data\\"

In [5]:
conds = pd.read_csv(clean + 'clean_condemnations.csv')

In [6]:
parcels = pd.read_csv(clean + 'blight.csv')

In [7]:
conds = conds.merge(parcels[['parcelID', 'lat', 'lng']], on = 'parcelID', how = 'left')

In [8]:
conds = conds.dropna()

In [10]:
fig = px.scatter_mapbox(conds, lat = 'lat', lon = 'lng', zoom = 10)
fig.update_layout(mapbox_style = 'open-street-map')
fig.show()

In [9]:
years = pd.date_range(start = conds['year'].min(), end = '2026-12-31', freq = 'Y')

In [10]:
parcels = parcels.merge(pd.DataFrame({'date': years}), how = 'cross').sort_values(by = 'date').reset_index(drop = True)
parcels['date'] = pd.to_datetime(parcels['date']).dt.year
parcels = parcels.rename(columns = {'date': 'year'})
years = range(conds['year'].min(), conds['year'].max() + 1)

In [11]:
full = pd.DataFrame()

earthRadius = 6_378_137 # meters

distances = np.arange(50, 800, 50)
distances_radians = distances / earthRadius
weights = [(i + 1) / len(distances) for i, r in enumerate(distances)][::-1]

for year in tqdm(years):
    parcels_sub = parcels.loc[parcels['year'] == year].reset_index(drop = True)
    values_sub = conds.loc[conds['year'] == year].reset_index(drop = True)

    parcels_coords = np.radians(parcels_sub[['lat', 'lng']].to_numpy())
    values_coords = np.radians(values_sub[['lat', 'lng']].to_numpy())

    value_tree = BallTree(values_coords, metric = 'haversine')
    
    # count conds in each distance bin
    for d_m, d_r in zip(distances, distances_radians):
        indices_within_radius = value_tree.query_radius(parcels_coords, r = d_r)
        counts = [len(idxs) for idxs in indices_within_radius]
        parcels_sub[f'conds_count_{d_m}'] = counts

    parcels_sub = parcels_sub.fillna(0)

    for i, r in enumerate(distances[::-1]):
        if r != distances[0]:
            parcels_sub[f'conds_count_{r}'] -= parcels_sub[f'conds_count_{distances[::-1][i + 1]}']

    parcels_sub['conds_count_decayed'] = (parcels_sub.iloc[:, -len(distances):] * weights).sum(axis = 1)

    q1 = parcels_sub['conds_count_decayed'].quantile(0.25)
    q3 = parcels_sub['conds_count_decayed'].quantile(0.75)
    iqr = q3 - q1
    lb = q1 - (1.5 * iqr)
    ub = q3 + (1.5 * iqr)

    parcels_sub['low_exposure'] = np.where(parcels_sub['conds_count_decayed'] < lb, 1, 0)
    parcels_sub['high_exposure'] = np.where(parcels_sub['conds_count_decayed'] > ub, 1, 0)

    full = pd.concat([full, parcels_sub], axis = 0)

full = full.reset_index(drop = True)

100%|██████████| 6/6 [00:52<00:00,  8.78s/it]


In [14]:
full.groupby('high_exposure')['blight'].value_counts(normalize = True)

high_exposure  blight
0              3        0.261
               2        0.238
               5        0.216
               1        0.113
               4        0.088
               6        0.070
               7        0.015
1              5        0.478
               6        0.333
               3        0.163
               2        0.017
               4        0.008
               1        0.000
Name: proportion, dtype: float64

In [ ]:
full['blight'].value_counts(normalize = True)

blight
3   0.255
5   0.231
2   0.225
1   0.106
6   0.086
4   0.083
7   0.014
Name: proportion, dtype: float64

In [9]:
full = pd.DataFrame()

earthRadius = 6_378_137 # meters

distances = np.arange(50, 800, 50)
distances_radians = distances / earthRadius
weights = [(i + 1) / len(distances) for i, r in enumerate(distances)][::-1]

parcels_sub = parcels.copy()
values_sub = conds.copy()

parcels_coords = np.radians(parcels_sub[['lat', 'lng']].to_numpy())
values_coords = np.radians(values_sub[['lat', 'lng']].to_numpy())

value_tree = BallTree(values_coords, metric = 'haversine')

# count conds in each distance bin
for d_m, d_r in zip(distances, distances_radians):
    indices_within_radius = value_tree.query_radius(parcels_coords, r = d_r)
    counts = [len(idxs) for idxs in indices_within_radius]
    parcels_sub[f'conds_count_{d_m}'] = counts

parcels_sub = parcels_sub.fillna(0)

for i, r in enumerate(distances[::-1]):
    if r != distances[0]:
        parcels_sub[f'conds_count_{r}'] -= parcels_sub[f'conds_count_{distances[::-1][i + 1]}']

parcels_sub['conds_count_decayed'] = (parcels_sub.iloc[:, -len(distances):] * weights).sum(axis = 1)

q1 = parcels_sub['conds_count_decayed'].quantile(0.25)
q3 = parcels_sub['conds_count_decayed'].quantile(0.75)
iqr = q3 - q1
lb = q1 - (1.5 * iqr)
ub = q3 + (1.5 * iqr)

parcels_sub['low_exposure'] = np.where(parcels_sub['conds_count_decayed'] < lb, 1, 0)
parcels_sub['high_exposure'] = np.where(parcels_sub['conds_count_decayed'] > ub, 1, 0)

full = pd.concat([full, parcels_sub], axis = 0)

full = full.reset_index(drop = True)

In [10]:
full.groupby('high_exposure')['blight'].value_counts(normalize = True)

high_exposure  blight
0              3        0.267
               2        0.235
               5        0.229
               1        0.111
               4        0.087
               6        0.057
               7        0.014
1              6        0.707
               5        0.289
               3        0.004
Name: proportion, dtype: float64

In [42]:
full['blight'].value_counts(normalize = True)

blight
3   0.255
5   0.231
2   0.225
1   0.106
6   0.086
4   0.083
7   0.014
Name: proportion, dtype: float64

In [51]:
full.groupby('blight')['conds_count_decayed'].median()

blight
1    1.933
2    5.200
3   14.200
4    5.267
5   35.267
6   32.133
7    3.400
Name: conds_count_decayed, dtype: float64

In [11]:
data = full[['blight', 'pc_blight', 'viols_sqrt', 'low_sqrt', 'high_sqrt', 'high_exposure', 'conds_count_decayed']]
data['conds_sqrt'] = np.sqrt(data['conds_count_decayed'])

In [12]:
for i in full['blight'].sort_values().unique():
    data[f'blight_{i}'] = np.where(data['blight'] == i, 1, 0)

In [13]:
data.corr().iloc[7:8, 1:]

,pc_blight,viols_sqrt,low_sqrt,high_sqrt,high_exposure,conds_count_decayed,conds_sqrt,blight_1,blight_2,blight_3,blight_4,blight_5,blight_6,blight_7
conds_sqrt,0.625,0.693,0.478,0.380,0.518,0.963,1.000,-0.360,-0.333,0.082,-0.166,0.419,0.329,-0.078


In [14]:
model = smf.ols(formula = 'conds_sqrt ~ C(blight)', data = data).fit()
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:             conds_sqrt   R-squared:                       0.471
Model:                            OLS   Adj. R-squared:                  0.471
Method:                 Least Squares   F-statistic:                 2.140e+04
Date:                Wed, 14 Jan 2026   Prob (F-statistic):               0.00
Time:                        16:28:20   Log-Likelihood:            -2.7996e+05
No. Observations:              144031   AIC:                         5.599e+05
Df Residuals:                  144024   BIC:                         5.600e+05
Df Model:                           6                                         
Covariance Type:            nonrobust                                         
                     coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------
Intercept          1.3813      0.014    101.

In [15]:
model = smf.ols(formula = 'conds_sqrt ~ viols_sqrt', data = data).fit()
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:             conds_sqrt   R-squared:                       0.481
Model:                            OLS   Adj. R-squared:                  0.481
Method:                 Least Squares   F-statistic:                 1.334e+05
Date:                Wed, 14 Jan 2026   Prob (F-statistic):               0.00
Time:                        16:28:27   Log-Likelihood:            -2.7865e+05
No. Observations:              144031   AIC:                         5.573e+05
Df Residuals:                  144029   BIC:                         5.573e+05
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     -0.3660      0.012    -29.895      0.0

In [124]:
model = smf.ols(formula = 'conds_sqrt ~ viols_sqrt + low_sqrt + high_sqrt', data = data).fit()
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:             conds_sqrt   R-squared:                       0.519
Model:                            OLS   Adj. R-squared:                  0.519
Method:                 Least Squares   F-statistic:                 5.175e+04
Date:                Mon, 12 Jan 2026   Prob (F-statistic):               0.00
Time:                        11:30:08   Log-Likelihood:            -2.7320e+05
No. Observations:              144031   AIC:                         5.464e+05
Df Residuals:                  144027   BIC:                         5.464e+05
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     -0.2444      0.013    -18.618      0.0

In [ ]:
full['conds_log'] = np.log1p(full['conds_count_decayed'])

In [18]:
model = smf.ols(formula = 'conds_log ~ viols_log', data = full).fit()
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:              conds_log   R-squared:                       0.543
Model:                            OLS   Adj. R-squared:                  0.543
Method:                 Least Squares   F-statistic:                 1.709e+05
Date:                Wed, 14 Jan 2026   Prob (F-statistic):               0.00
Time:                        16:29:21   Log-Likelihood:            -1.6619e+05
No. Observations:              144031   AIC:                         3.324e+05
Df Residuals:                  144029   BIC:                         3.324e+05
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     -3.4049      0.014   -237.508      0.0

In [17]:
model = smf.ols(formula = 'conds_log ~ viols_log + low_log + high_log', data = full).fit()
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:              conds_log   R-squared:                       0.587
Model:                            OLS   Adj. R-squared:                  0.587
Method:                 Least Squares   F-statistic:                 6.816e+04
Date:                Wed, 14 Jan 2026   Prob (F-statistic):               0.00
Time:                        16:29:14   Log-Likelihood:            -1.5888e+05
No. Observations:              144031   AIC:                         3.178e+05
Df Residuals:                  144027   BIC:                         3.178e+05
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     -3.7344      0.014   -268.697      0.0

In [23]:
data[['conds_sqrt', 'viols_sqrt', 'low_sqrt', 'high_sqrt']].corr().iloc[:1, 1:]

,viols_sqrt,low_sqrt,high_sqrt
conds_sqrt,0.693,0.478,0.380


In [22]:
full[['conds_log', 'viols_log', 'low_log', 'high_log']].corr().iloc[:1, 1:]

,viols_log,low_log,high_log
conds_log,0.737,0.537,0.453


In [27]:
pearsonr(data['conds_sqrt'], data['viols_sqrt'])

PearsonRResult(statistic=np.float64(0.693448008826648), pvalue=np.float64(0.0))

### Argument

The condemnation is a description of the blight, not a cause or factor of the blight.

In [16]:
full.groupby('blight_sqrt')['high_exposure'].mean()

blight_sqrt
1   0.000
2   0.004
3   0.033
4   0.049
5   0.153
6   0.303
7   0.000
Name: high_exposure, dtype: float64

In [17]:
full['high_exposure'].mean()

np.float64(0.060419863316461965)

In [18]:
dist_pop = full['blight_sqrt'].value_counts(normalize = True)
n_pop = full['parcelID'].nunique() # 6 years represented, so this reduces that to one count per parcel

sample = full.loc[full['high_exposure'] == 1]
dist_samp = sample['blight_sqrt'].value_counts(normalize = True)

for i in range(1, 8):
    se = np.sqrt( (dist_pop[i]*(1 - dist_pop[i])) / n_pop )

    z = (dist_samp[i] - dist_pop[i]) / se

    p = 2 * (1 - norm.cdf(abs(z)))

    print(f'Blight: {i}:   Z-score: {round(z, 2)},   P-value: {round(p, 2)}')

Blight: 1:   Z-score: -168.11,   P-value: 0.0
Blight: 2:   Z-score: -193.76,   P-value: 0.0
Blight: 3:   Z-score: -97.95,   P-value: 0.0
Blight: 4:   Z-score: -28.19,   P-value: 0.0
Blight: 5:   Z-score: 228.4,   P-value: 0.0
Blight: 6:   Z-score: 450.74,   P-value: 0.0


KeyError: 7

In [19]:
(
    full
    .groupby('blight_sqrt').agg({
    'parcelID': 'count',
    'conds_count_decayed': 'sum'
    })
    .assign(
        conds_per_parcel = lambda x: x['conds_count_decayed'] / x['parcelID']
    )
)

,parcelID,conds_count_decayed,conds_per_parcel
blight_sqrt,,,
1,143358,64239.133,0.448
2,197046,231943.133,1.177
3,208368,643315.267,3.087
4,118110,395882.733,3.352
5,115818,801810.933,6.923
6,69732,716976.467,10.282
7,11754,9848.667,0.838


In [20]:
full['conds_count_decayed'].sum() / full['parcelID'].count()

np.float64(3.3141202626903628)

In [115]:
mean = parcels['pc_blight'].mean()
std = parcels['pc_blight'].std()

extremes = parcels.loc[(parcels['pc_blight'] > (mean + 2*std)) | (parcels['pc_blight'] < (mean - 2*std))]

In [122]:
extremes.loc[extremes['blight'] == 6]['nbrhd'].value_counts()

nbrhd
South Side Flats     2320
Allentown             896
South Side Slopes     471
Homewood North        456
Knoxville             439
Central Oakland       301
Homewood South        228
Beltzhoover            62
South Oakland           8
Mount Washington        3
Name: count, dtype: int64

In [ ]:
median = parcels['pc_blight'].median()
mad = (parcels['pc_blight'] - median).abs().median()
outliers = parcels.loc[
    ((parcels['pc_blight'] - median).abs() / mad) > 3
]

In [120]:
outliers['blight'].value_counts()

blight
6    5848
1    2336
7     552
Name: count, dtype: int64